# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. We focus on referencing all dataset entities (record sets, fields, columns, etc.) by their unique `@id` identifiers, in line with best practices for working with Croissant schemas.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata object
dataset = mlc.Dataset(croissant_url)

# Print main metadata fields
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}\nPublished: {getattr(metadata, 'datePublished', 'N/A')}")

## 2. Data Overview
Review available record sets (tables), fields (columns), and their unique `@id`s. All references below use the Croissant `@id` for traceability and reproducibility.

First, enumerate all available record sets. For each record set, list its available fields and their `@id`s.

In [ ]:
# List all record sets by @id (Croissant representation)
print("Available record sets in the dataset:")
for record_set in dataset.record_sets:
    print(f"- Record Set Name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields (columns):")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, dtype: {field.data_type})")
    print()

## 3. Data Extraction
Below, we extract data from a chosen record set into a DataFrame for analysis. All variable assignments (record set and fields) use their unique `@id` values as demonstrated above.

For this analysis, we pick the main record set (first listed) for further exploration. Adjust variables as needed to explore other record sets.

In [ ]:
# Choose the record set by @id (use the printed output above to select others)
# Example: main data table
main_record_set_id = dataset.record_sets[0].id
record_sets_ids = [rs.id for rs in dataset.record_sets]

# Load all selected record sets into DataFrames
dataframes = {}
for rs_id in record_sets_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

print(f"Loaded data columns for main record set (@id: {main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())

# Show first few records
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's analyze the distributions and apply basic processing steps. We'll select a numeric field (referenced by its `@id`) for filtering and normalization. 

_You may adjust `numeric_field_id` and `group_field_id` below based on the printed columns and field ids from prior steps._

In [ ]:
# Assign the @id of a numeric field and a group field from the main record set
# Example (replace with your actual field @ids from cell [5]):
# numeric_field_id = 'cr:field/abundance'  # Adjust according to actual field
# group_field_id = 'cr:field/sample'      # Adjust according to actual field

# For demonstration, let's pick the first field detected as 'Float' or 'Integer'
main_record_set = dataset.record_set(main_record_set_id)
numeric_field_id = None
for field in main_record_set.fields:
    if field.data_type in ['Float', 'Integer', 'Number']:
        numeric_field_id = field.id
        break

print(f"Using numeric field: {numeric_field_id}")

# Try to find a group field that's string-like
group_field_id = None
for field in main_record_set.fields:
    if field.data_type == 'Text' and field.id != numeric_field_id:
        group_field_id = field.id
        break
print(f"Group by field: {group_field_id}")

df = dataframes[main_record_set_id]
if numeric_field_id in df.columns:
    # Remove NA and convert to float just in case
    df = df.copy()
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f'Filtered records with {numeric_field_id} > mean (≈{threshold:.2f}): {len(filtered_df)} records')
    
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Optional: group by a group field
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame(name='mean')
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print(f"Numeric field {numeric_field_id} not found in dataframe columns.")

## 5. Visualization
Visualize distributions or relationships. Here we plot the distribution of the selected numeric field, and optionally a grouped bar chart if grouping is available.

In [ ]:
import matplotlib.pyplot as plt

if numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouped_df available, show bar plot
    if 'grouped_df' in locals():
        grouped_df.head(10).plot(kind='bar', legend=False)
        plt.title(f"Mean {numeric_field_id} by {group_field_id} (Top 10)")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.tight_layout()
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to load and explore a dataset defined by a Croissant schema using the `mlcroissant` library. By referencing all entities via their `@id`, we ensured reproducibility and clarity. We identified available record sets and fields, extracted and processed data, and conducted exploratory data analysis, including grouping and visualization.

- For robust analysis, always reference entities by their Croissant `@id`.
- The `mlcroissant` approach supports FAIR, machine-actionable workflows.